# seed_silver_test_data

**Source:** `06_analysis/seed_silver_test_data.py`  
**Purpose:** Databricks notebook auto-generated from framework Python module.


## Section 1: Additional module code and configuration

This cell handles: *Additional module code and configuration*


In [ ]:
"""Seed deterministic test rows into a silver test table from an existing silver source table."""

from __future__ import annotations

import uuid
from datetime import datetime, timezone
from typing import Any


## Section 2: Define `_as_bool()` helper function

This cell handles: *Define `_as_bool()` helper function*


In [ ]:
def _as_bool(value: str) -> bool:
    return str(value).strip().lower() in {"1", "true", "t", "yes", "y"}


## Section 3: Define `_fqn()` helper function

This cell handles: *Define `_fqn()` helper function*


In [ ]:
def _fqn(catalog: str, schema: str, table: str) -> str:
    return f"{catalog}.{schema}.{table}"


## Section 4: Define `_seed_rows()` function with logic for processing

This cell handles: *Define `_seed_rows()` function with logic for processing*


In [ ]:
def _seed_rows(spark, source_table_fqn: str, target_table_fqn: str, row_limit: int, mode: str):
    seed_run_id = uuid.uuid4().hex
    seed_ts = datetime.now(timezone.utc).isoformat()

    source_df = spark.table(source_table_fqn).limit(row_limit)
    source_df.createOrReplaceTempView("_seed_input")

    spark.sql(
        f"""
        CREATE TABLE IF NOT EXISTS {target_table_fqn}
        USING DELTA
        AS
        SELECT *,
               '{seed_run_id}' AS _seed_run_id,
               '{seed_ts}' AS _seed_ts
        FROM _seed_input
        WHERE 1 = 0
        """
    )

    if mode == "overwrite":
        spark.sql(f"TRUNCATE TABLE {target_table_fqn}")

    spark.sql(
        f"""
        INSERT INTO {target_table_fqn}
        SELECT *,
               '{seed_run_id}' AS _seed_run_id,
               '{seed_ts}' AS _seed_ts
        FROM _seed_input
        """
    )

    row_count = spark.sql(f"SELECT COUNT(*) AS c FROM {target_table_fqn}").collect()[0]["c"]
    print(f"seed_run_id={seed_run_id}")
    print(f"seed_ts={seed_ts}")
    print(f"target_row_count={row_count}")


_dbutils: Any = globals().get("dbutils")
_spark: Any = globals().get("spark")

if _dbutils is None:
    raise RuntimeError("This notebook must run in Databricks where dbutils is available")

_dbutils.widgets.text("execute", "false")
_dbutils.widgets.text("catalog", "main")
_dbutils.widgets.text("silver_schema", "silver_dev")
_dbutils.widgets.text("source_table", "connect_countryriskdet")
_dbutils.widgets.text("target_table", "connect_countryriskdet_test_seed")
_dbutils.widgets.text("row_limit", "100")
_dbutils.widgets.text("mode", "overwrite")

execute = _as_bool(_dbutils.widgets.get("execute"))
catalog = _dbutils.widgets.get("catalog").strip()
silver_schema = _dbutils.widgets.get("silver_schema").strip()
source_table = _dbutils.widgets.get("source_table").strip()
target_table = _dbutils.widgets.get("target_table").strip()
row_limit = int(_dbutils.widgets.get("row_limit").strip() or "100")
mode = _dbutils.widgets.get("mode").strip().lower() or "overwrite"

source_fqn = _fqn(catalog, silver_schema, source_table)
target_fqn = _fqn(catalog, silver_schema, target_table)

print("execute=", execute)
print("source=", source_fqn)
print("target=", target_fqn)
print("row_limit=", row_limit)
print("mode=", mode)

if not execute:
    print("Skipping seed run because execute=false")
else:
    if mode not in {"overwrite", "append"}:
        raise ValueError("mode must be overwrite or append")
    if _spark is None:
        raise RuntimeError("This notebook must run in Databricks where spark is available")
    _seed_rows(_spark, source_fqn, target_fqn, row_limit=row_limit, mode=mode)
